# WFS and GeoJSON API with GeoPandas

In this notebook you will learn how to consume geospatial data from web services and load it in Python with `geopandas`, using schools in Aragon as a real case study.

## Objectives

- Understand the difference between a **WFS** service and a **GeoJSON API**.
- Load remote geospatial layers into a `GeoDataFrame`.
- Explore, visualize, and filter spatial data using a real education dataset.
- Export a selection to GeoJSON for reuse.


## 1) Key Concepts

### WFS (Web Feature Service)
- OGC standard for querying **vector features** (points, lines, polygons).
- It usually allows filtering by attributes, spatial extent, and output format.
- Very common in institutional spatial data infrastructures.

### GeoJSON API
- Web API that directly returns objects in **GeoJSON** format.
- It is often simpler to consume from Python and front-end web applications.
- It does not always follow the OGC standard, but it is very practical.

> In practice, both approaches let you bring geometry + attributes into a `GeoDataFrame`.


In [ ]:
# If a dependency is missing, uncomment this cell
# %pip install geopandas requests matplotlib pyogrio

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['axes.grid'] = True

## 2) Example A: Load data from a GeoJSON API

We will use an Open Data Aragon endpoint with information about education centers.

> Note: the service may return different types of centers. In the analysis, we will specifically filter public schools.


In [ ]:
geojson_url = "https://opendata.aragon.es/GA_OD_Core/download?view_id=167&formato=json"

gdf_geojson = gpd.read_file(geojson_url)

# The service declares EPSG:4326, but the coordinates are in UTM zone 30N.
# Assign the correct CRS before reprojecting or combining with other layers.
gdf_geojson = gdf_geojson.set_crs(epsg=25830, allow_override=True)

gdf_geojson.head()

In [ ]:
print("CRS:", gdf_geojson.crs)
print("Number of features:", len(gdf_geojson))
print("Main columns:", ["nombre_cen", "tipo_centr", "naturaleza", "localidad", "geometry"])

# Quick view of key fields
gdf_geojson[["nombre_cen", "tipo_centr", "naturaleza", "localidad", "geometry"]].head()

In [ ]:
ax = gdf_geojson.plot(edgecolor="black", alpha=0.6, markersize=20)
ax.set_title("Education centers in Aragon (GeoJSON API)")
ax.set_axis_off()

In [ ]:
gdf_geojson["naturaleza"]=="Público"

## 3) Cleaning and Filtering: Public Schools in Aragon

The dataset includes different types of centers. We will keep centers whose `naturaleza` value is public and whose type contains "Colegio Publico".

This builds a more realistic example of attribute selection.


In [ ]:
gdf_colegios_publicos = gdf_geojson[
    (gdf_geojson["naturaleza"].str.contains("Público", case=False, na=False))
].copy()
# equivalent: gdf_geojson[gdf_geojson["naturaleza"]=="Público"].copy()

gdf_colegios_publicos[["nombre_cen", "tipo_centr", "naturaleza", "localidad", "geometry"]].head()

In [ ]:
print("CRS:", gdf_colegios_publicos.crs)
print("Number of public schools:", len(gdf_colegios_publicos))
print("Number of represented localities:", gdf_colegios_publicos["localidad"].nunique())

In [ ]:
ax = gdf_colegios_publicos.plot(color="royalblue", edgecolor="black", markersize=18)
ax.set_title("Public schools in Aragon")
ax.set_axis_off()

## 4) Quick Analysis by Locality

A common teaching question is: "which localities have the most public schools?"

We can answer it with a count by `localidad`.


In [ ]:
top_localidades = (
    gdf_colegios_publicos.groupby("localidad")
    .size()
    .sort_values(ascending=False)
    .reset_index(name="num_colegios")
)

top_localidades.head(10)

In [ ]:
ax = top_localidades.head(10).plot.bar(x="localidad", y="num_colegios", legend=False)
ax.set_title("Top 10 localities by number of public schools")
ax.set_xlabel("Locality")
ax.set_ylabel("Number of schools")

## 5) Export the Result to GeoJSON

Saving the public-school selection is useful for sharing results or using them in web viewers.


In [ ]:
salida_geojson = "data/colegios_publicos_aragon.geojson"
gdf_colegios_publicos.to_file(salida_geojson, driver="GeoJSON")
print(f"Exported file: {salida_geojson}")

## 6) Example B: Load Data from WFS (IDEE Transport)

Now we use a real WFS service from IDEE/IGN:

- Service: `https://servicios.idee.es/wfs-inspire/transportes`
- Operation: `request=GetFeature`
- Example layer: `tn-ro:RoadLink`

This service usually returns GML, which `geopandas` can read directly.


In [ ]:
# Fixed WFS example with the requested layer: tn-ro:RoadLink
# To keep the example fast and readable, use a specific urban area.
# These coordinates are in EPSG:25830, the CRS of the schools.
colegios_zaragoza = gdf_colegios_publicos[
    gdf_colegios_publicos["localidad"].str.contains("Zaragoza", case=False, na=False)
].cx[670000:681000, 4609000:4620000].copy()

# Build a bbox from those schools, with a small margin in degrees.
xmin, ymin, xmax, ymax = colegios_zaragoza.to_crs(4326).total_bounds
margen = 0.01
bbox_escuelas = f"{xmin-margen},{ymin-margen},{xmax+margen},{ymax+margen},EPSG:4326"

url_wfs_aragon = (
    "https://servicios.idee.es/wfs-inspire/transportes?"
    "service=WFS&version=2.0.0&request=GetFeature&"
    "typeNames=tn-ro:RoadLink&count=3000&"
    f"srsName=EPSG:4326&bbox={bbox_escuelas}"
)

gdf_wfs_aragon = gpd.read_file(url_wfs_aragon)
print("Public schools in Zaragoza:", len(colegios_zaragoza))
print("WFS features loaded:", len(gdf_wfs_aragon))
print("BBox used:", bbox_escuelas)
gdf_wfs_aragon.head()

In [ ]:
len(gdf_wfs_aragon) #! Look the number of entities. Are they all in Aragon?

In [ ]:
# Plot the data
ax = gdf_wfs_aragon.plot(color="darkgreen", linewidth=4)
ax.set_title("WFS IDEE Transportes - tn-ro:RoadLink")
ax.set_axis_off()
#! Look at the data


## 8) Example C: Operations on Vector Data

In this block, we combine layers, apply spatial operations, and calculate basic metrics for territorial analysis.


In [ ]:
# C.2 Combine two GeoDataFrames: public schools + road network
# Use a metric CRS so distances and lengths are in meters.
crs_metrico = 25830

colegios_m = colegios_zaragoza.to_crs(epsg=crs_metrico)
red_m = gdf_wfs_aragon.to_crs(epsg=crs_metrico)

# Clean null or empty geometries before operating.
colegios_m = colegios_m[colegios_m.geometry.notna() & ~colegios_m.geometry.is_empty].copy()
red_m = red_m[red_m.geometry.notna() & ~red_m.geometry.is_empty].copy()

# Clip the network to the area occupied by the schools, with a 1 km margin.
xmin, ymin, xmax, ymax = colegios_m.total_bounds
margen_m = 1000
red_recortada = red_m.cx[xmin-margen_m:xmax+margen_m, ymin-margen_m:ymax+margen_m].copy()
red_recortada["longitud_m"] = red_recortada.geometry.length

print("Public schools in the area:", len(colegios_m))
print("Road segments in the area:", len(red_recortada))
print("Total network length (km):", round(red_recortada["longitud_m"].sum() / 1000, 2))

fig, ax = plt.subplots(figsize=(10, 8))

if red_recortada.empty:
    colegios_m.plot(ax=ax, color="crimson", markersize=18, alpha=0.9, label="Public schools")
    ax.set_title("Public schools in Zaragoza")
else:
    red_recortada.plot(ax=ax, color="steelblue", linewidth=1, alpha=0.75, label="Road network")
    colegios_m.plot(ax=ax, color="crimson", edgecolor="white", markersize=24, alpha=0.9, label="Public schools")
    ax.set_title("Public schools and road network in Zaragoza")

ax.legend()
ax.set_axis_off()

## 9) Proposed Activities

1. Filter only centers from a specific province.
2. Reproject the data to EPSG:3857 and plot it again.
3. In the WFS, try another `typeNames` value from `GetCapabilities`.
4. Compare the column schema and geometry between GeoJSON and WFS.

## Closing

With `geopandas`, you can integrate remote data (GeoJSON API or WFS) into spatial analysis workflows directly and reproducibly.